# DPO
将大型语言模型（LLMs）与人类价值观和偏好对齐是一项挑战。传统方法，如人类反馈的强化学习（RLHF），通过整合人类输入来优化模型输出。然而，RLHF 可能复杂且资源密集，要求大量的计算能力和数据处理。直接偏好优化（DPO）作为一种新颖且更简化的方法出现，为这些传统方法提供了一种高效的替代方案。通过简化优化过程，DPO 不仅减少了计算负担，还增强了模型快速适应人类偏好的能力。

## 偏好对齐的重要性
要了解 DPO，首先必须理解为何将LLMs与人类偏好对齐如此重要。尽管LLMs在庞大的数据集上训练具有令人印象深刻的能力，但有时可能会产生与人类价值观不一致、有偏见或不对齐的输出。这种不对齐可能以多种方式表现出来：
- 生成不安全或有害的内容
- 提供不准确或误导性信息
- 展示训练数据中存在的偏见

为了应对这些问题，研究人员开发了利用人类反馈来微调LLMs的技术。其中最突出的方式是 RLHF。

## RLHF
![image.png](./_img/chatgpt_rlhf.png)

基于人类反馈的强化学习（RLHF）一直是使LLMs与人类偏好一致的首选方法。让我们分解 RLHF 过程以理解其复杂性：

1. 监督微调（SFT）：该过程首先在高质量回答的数据集上微调一个预训练的LLM。这一步有助于模型为目标任务生成更相关和连贯的输出。
2. 奖励建模：训练一个单独的奖励模型来预测人类偏好。这包括：
- 为给定提示生成响应对
- 让人类评估他们更喜欢哪种回复
- 训练模型以预测这些偏好

3. 强化学习：微调后的 LLM 接着通过强化学习进一步优化。奖励模型提供反馈，引导 LLM 生成与人类偏好一致的响应。

### RLHF的缺点
- 它需要训练和维护多个模型（SFT、奖励模型和 RL 优化模型）
- RL 过程可能不稳定且对超参数敏感
- 这在计算上是昂贵的，需要在模型中进行多次前向和反向传递

### DPO核心概念
![Alt text](./_img/DPO_framework.png)
这张图片对比了两种不同的方法，将LLM输出与人类偏好对齐：来自人类反馈的强化学习（RLHF）和直接偏好优化（DPO）。RLHF 依赖一个奖励模型，通过迭代反馈循环指导语言模型的策略，而 DPO 则使用偏好数据直接优化模型输出，以匹配人类偏好的响应。这种比较突出了每种方法的优势和潜在应用，为未来LLMs的训练提供了见解，以更好地与人类期望对齐。

### 关键思想
1. **Implicit Reward Modeling**：隐式奖励建模：DPO 通过将语言模型本身视为隐式奖励函数，消除了对单独奖励模型的需求。
2. **Policy-Based Formulation**：DPO 直接优化策略（语言模型），以最大化偏好响应的概率，而不是优化奖励函数。
3. **Closed-Form Solution**：DPO 利用一种数学洞察，允许对最优策略进行封闭形式解，从而避免了迭代强化学习更新的需要。

### 损失函数
下面的图像展示了使用 PyTorch 实现 DPO 损失函数的代码片段。这个函数在优化语言模型根据人类偏好优先输出方面发挥了关键作用。以下是关键组件的分解：

- **Function Signature**： `dpo_loss` 函数接受多个参数，包括策略日志概率 ( `pi_logps` )、参考模型日志概率 ( `ref_logps` ) 和表示首选和非首选完成的索引 ( `yw_idxs` 、 `yl_idxs` )。此外， `beta` 参数控制 KL 惩罚的强度。
- **Log Probability Extraction**：该代码从政策模型和参考模型中提取首选和不首选完成的对数概率。
- **Log Ratio Calculation**：优选和非优选完成的对数概率之间的差异对于策略模型和参考模型均进行计算。该比率在确定优化的方向和大小方面至关重要。
- **Loss and Reward Calculation**：损失通过 `logsigmoid` 函数计算，而奖励是通过将策略和参考对数概率之间的差异乘以 `beta` 来确定的。

In [ ]:
import torch.nn.functional as F
def dpo_loss(pi_logps, ref_logps, yw_idxs, yl_idxs, beta):
    """
    pi_logps: policy logprobs, shape (B,)
    ref_logps: reference model logprobs, shape (B,)
    yw_idxs: preferred completion indices in [0, B-1], shape (T,)
    yl_idxs: dispreferred completion indices in [0, B-1], shape (T,)
    beta: temperature controlling strength of KL penalty
    Each pair of (yw_idxs[i], yl_idxs[i]) represents the
    indices of a single preference pair.
    """
    pi_yw_logps, pi_yl_logps = pi_logps[yw_idxs], pi_logps[yl_idxs]
    ref_yw_logps, ref_yl_logps = ref_logps[yw_idxs], ref_logps[yl_idxs]
    pi_logratios = pi_yw_logps- pi_yl_logps
    ref_logratios = ref_yw_logps- ref_yl_logps
    losses =-F.logsigmoid(beta * (pi_logratios- ref_logratios))
    rewards = beta * (pi_logps- ref_logps).detach()
    return losses, rewards

### DPO数学公式
DPO 是偏好学习问题的一种巧妙重新表述。这是逐步分析：
1. Starting Point: KL-约束奖励最大化
> 原始 RLHF 目标可以表示为：
> ![Alt text](./_img/RLHF_f1.png)
> 其中：
> - $πθ$ 是我们正在优化的策略（语言模型）
> - $r(x,y)$是奖励函数
> - $πref$ 是一个参考策略（通常是初始 SFT 模型）
> - $β$ 控制 KL 散度约束的强度

2. 最优政策形式：可以证明，该目标的最优政策采取以下形式：
> $ π_r(y|x) = 1/Z(x) * πref(y|x) * exp(1/β * r(x,y))v$
> 其中 $Z(x)$ 是一个归一化常数。

3.  **Reward-Policy Duality:**DPO 的关键观点是通过最优策略来表达奖励函数：
> $r(x,y) = β * log(π_r(y|x) / πref(y|x)) + β * log(Z(x))$

4. 偏好模型 假设偏好遵循布拉德利-特里模型，我们可以将更喜欢 y1 而不是 y2 的概率表示为：
> $p*(y1 ≻ y2 | x) = σ(r*(x,y1) - r*(x,y2))$
> 其中 $σ$ 是逻辑函数。

5. **DPO Objective**: DPO 目标 将我们的奖励政策二元性替代到偏好模型中，我们得出 DPO 目标：
> $L_DPO(πθ; πref) = -E_(x,y_w,y_l)~D [log σ(β * log(πθ(y_w|x) / πref(y_w|x)) - β * log(πθ(y_l|x) / πref(y_l|x)))]$
>
> 这个目标可以使用标准的梯度下降技术进行优化，无需使用强化学习算法。

## DPO代码实现

In [ ]:
import torch
import torch.nn.functional as F
class DPOTrainer:
    def __init__(self, model, ref_model, beta=0.1, lr=1e-5):
        self.model = model
        self.ref_model = ref_model
        self.beta = beta
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
     
    def compute_loss(self, pi_logps, ref_logps, yw_idxs, yl_idxs):
        """
        pi_logps: policy logprobs, shape (B,)
        ref_logps: reference model logprobs, shape (B,)
        yw_idxs: preferred completion indices in [0, B-1], shape (T,)
        yl_idxs: dispreferred completion indices in [0, B-1], shape (T,)
        beta: temperature controlling strength of KL penalty
        Each pair of (yw_idxs[i], yl_idxs[i]) represents the indices of a single preference pair.
        """
        # Extract log probabilities for the preferred and dispreferred completions
        pi_yw_logps, pi_yl_logps = pi_logps[yw_idxs], pi_logps[yl_idxs]
        ref_yw_logps, ref_yl_logps = ref_logps[yw_idxs], ref_logps[yl_idxs]
        # Calculate log-ratios
        pi_logratios = pi_yw_logps - pi_yl_logps
        ref_logratios = ref_yw_logps - ref_yl_logps
        # Compute DPO loss
        losses = -F.logsigmoid(self.beta * (pi_logratios - ref_logratios))
        rewards = self.beta * (pi_logps - ref_logps).detach()
        return losses.mean(), rewards
    def train_step(self, batch):
        x, yw_idxs, yl_idxs = batch
        self.optimizer.zero_grad()
        # Compute log probabilities for the model and the reference model
        pi_logps = self.model(x).log_softmax(-1)
        ref_logps = self.ref_model(x).log_softmax(-1)
        # Compute the loss
        loss, _ = self.compute_loss(pi_logps, ref_logps, yw_idxs, yl_idxs)
        loss.backward()
        self.optimizer.step()
        return loss.item()
# Usage
model = YourLanguageModel()  # Initialize your model
ref_model = YourLanguageModel()  # Load pre-trained reference model
trainer = DPOTrainer(model, ref_model)
for batch in dataloader:
    loss = trainer.train_step(batch)
    print(f"Loss: {loss}")

## 未来方向
尽管 DPO 相对于传统的 RLHF 方法具有显著优势，但仍然存在挑战和进一步研究的领域：

### 可扩展到更大的模型
随着语言模型规模的不断扩大，如何高效地将 DPO 应用于拥有数千亿参数的模型仍然是一个未解的挑战。研究人员正在探索以下技术：
- 高效微调方法（例如，LoRA，前缀调优）
- 分布式训练优化
- 梯度检查点和混合精度训练

使用LoRA和DPO的示例：

In [ ]:
from peft import LoraConfig, get_peft_model
class DPOTrainerWithLoRA(DPOTrainer):
    def __init__(self, model, ref_model, beta=0.1, lr=1e-5, lora_rank=8):
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=32,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        self.model = get_peft_model(model, lora_config)
        self.ref_model = ref_model
        self.beta = beta
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
# Usage
base_model = YourLargeLanguageModel()
dpo_trainer = DPOTrainerWithLoRA(base_model, ref_model)

### 多任务和少样本适应
开发能够有效适应有限偏好数据的新任务或领域的 DPO 技术是一个活跃的研究领域。正在探索的方法包括：
-  快速适应的元学习框架
- DPO 的基于提示的微调
- 从一般偏好模型到特定领域的迁移学习

### 处理模糊或冲突的偏好：
现实世界的偏好数据通常包含模糊性或冲突。提高 DPO 对这类数据的鲁棒性至关重要。潜在的解决方案包括：
- 概率偏好建模
- 主动学习以解决歧义
- 多主体偏好聚合

概率偏好建模的示例：

In [ ]:
class ProbabilisticDPOTrainer(DPOTrainer):
    def compute_loss(self, pi_logps, ref_logps, yw_idxs, yl_idxs, preference_prob):
        # Compute log ratios
        pi_yw_logps, pi_yl_logps = pi_logps[yw_idxs], pi_logps[yl_idxs]
        ref_yw_logps, ref_yl_logps = ref_logps[yw_idxs], ref_logps[yl_idxs]
         
        log_ratio_diff = pi_yw_logps.sum(-1) - pi_yl_logps.sum(-1)
        loss = -(preference_prob * F.logsigmoid(self.beta * log_ratio_diff) +
                 (1 - preference_prob) * F.logsigmoid(-self.beta * log_ratio_diff))
        return loss.mean()
# Usage
trainer = ProbabilisticDPOTrainer(model, ref_model)
loss = trainer.compute_loss(pi_logps, ref_logps, yw_idxs, yl_idxs, preference_prob=0.8)  # 80% confidence in preference


### 将 DPO 与其他对齐技术结合
将 DPO 与其他对齐方法集成可能会导致更强大和更有能力的系统
- 明确约束满足的宪法人工智能原则
- 复杂偏好引导的辩论与递归奖励建模
- 用于推断潜在奖励函数的逆向强化学习

将 DPO 与宪法 AI 结合的示例：

In [ ]:
class ConstitutionalDPOTrainer(DPOTrainer):
    def __init__(self, model, ref_model, beta=0.1, lr=1e-5, constraints=None):
        super().__init__(model, ref_model, beta, lr)
        self.constraints = constraints or []
    def compute_loss(self, pi_logps, ref_logps, yw_idxs, yl_idxs):
        base_loss = super().compute_loss(pi_logps, ref_logps, yw_idxs, yl_idxs)
         
        constraint_loss = 0
        for constraint in self.constraints:
            constraint_loss += constraint(self.model, pi_logps, ref_logps, yw_idxs, yl_idxs)
         
        return base_loss + constraint_loss
# Usage
def safety_constraint(model, pi_logps, ref_logps, yw_idxs, yl_idxs):
    # Implement safety checking logic
    unsafe_score = compute_unsafe_score(model, pi_logps, ref_logps)
    return torch.relu(unsafe_score - 0.5)  # Penalize if unsafe score > 0.5
constraints = [safety_constraint]
trainer = ConstitutionalDPOTrainer(model, ref_model, constraints=constraints)